In [ ]:
# Installation of ifcopenshell
!pip install ifcopenshell

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.8/40.8 MB 15.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.0/111.0 kB 5.7 MB/s eta 0:00:00


In [ ]:
# Identifies walls without properties and adds them.
# Usecase: revit can't export instance properties for stacked walls.
# Version: 0.1 - 03.07.2025 - IFC4x3

import ifcopenshell
import ifcopenshell.util.element
import uuid
import os # Import os module to manipulate file paths

# Loading the ifc file
input_filename = 'LU32_AR_MD_ME_LEITM.ifc'
ifc_file = ifcopenshell.open(input_filename)

# Creating new property values (these will be templates to be added per wall)
property_arealid_template = ifc_file.create_entity(
    "IfcPropertySingleValue",
    "ArealID",
    None,  # Description is optional
    ifc_file.create_entity("IfcText", "LU"),  # Nominal value
    None  # Unit is optional
)

property_gebaeudeid_template = ifc_file.create_entity(
    "IfcPropertySingleValue",
    "GebaeudeID",
    None,  # Description is optional
    ifc_file.create_entity("IfcText", "LU32"),  # Nominal value
    None  # Unit is optional
)

property_unterhaltsrelevant_template = ifc_file.create_entity(
    "IfcPropertySingleValue",
    "Unterhaltsrelevant",
    None,  # Description is optional
    ifc_file.create_entity("IfcText", "nein"),  # Nominal value
    None  # Unit is optional
)

# Note: GeschossID will be created per wall with the specific storey value


# search for walls that don't have property ArealID
walls_without_arealid = []
walls = ifc_file.by_type("IfcWall") # Assuming walls are of type IfcWall. Adjust if necessary.

for wall in walls:
    has_arealid = False
    if hasattr(wall, 'IsDefinedBy'):
        for rel in wall.IsDefinedBy:
            if rel.is_a('IfcRelDefinesByProperties'):
                property_set = rel.RelatingPropertyDefinition
                if property_set.is_a('IfcPropertySet'):
                    for prop in property_set.HasProperties:
                        if prop.is_a('IfcPropertySingleValue') and prop.Name == 'ArealID':
                            has_arealid = True
                            break  # Found ArealID, move to the next wall
    if not has_arealid:
        walls_without_arealid.append(wall)

# You can now work with the walls_without_arealid list
modified_wall_count = 0

# Function to find the Building Storey for an element by searching relationships
def find_building_storey(element):
    # Search for IfcRelContainedInSpatialStructure relationships that contain this element
    contained_rels = ifc_file.by_type("IfcRelContainedInSpatialStructure")
    for rel in contained_rels:
        if hasattr(rel, 'RelatedElements') and element in rel.RelatedElements:
            if hasattr(rel, 'RelatingStructure'):
                relating_structure = rel.RelatingStructure
                # We are looking for a Building Storey
                if relating_structure.is_a('IfcBuildingStorey'):
                    return relating_structure
                else:
                    # If the relating structure is not a storey, we might need to traverse up
                    # For simplicity now, we focus on direct containment in Storey via this relationship
                    pass # No recursion here in this simplified fallback

    # If not found in any relevant relationship, try the placement hierarchy as a fallback (less reliable)
    if hasattr(element, 'ObjectPlacement'):
        current_placement = element.ObjectPlacement
        while current_placement:
            if hasattr(current_placement, 'PlacementRelTo'):
                placement_rel_to = current_placement.PlacementRelTo

                # Try to find the entity that the placement is relative to
                if hasattr(placement_rel_to, 'GlobalId'):
                     related_entity = ifc_file.by_global_id(placement_rel_to.GlobalId)
                     if related_entity:
                         if related_entity.is_a('IfcBuildingStorey'):
                             return related_entity # Found the containing Building Storey via placement
                         else:
                             # Move up the placement hierarchy
                             current_placement = placement_rel_to # Update current_placement for the next iteration
                             continue # Continue the while loop


            # If no PlacementRelTo or not found, stop traversing this branch
            current_placement = None
    return None # Storey not found


# Iterate through the walls without ArealID and add/update the property set
for wall in walls_without_arealid:
    wall_pset_luks = None

    # Check if the wall already has a Pset_LUKS
    if hasattr(wall, 'IsDefinedBy'):
        for rel in wall.IsDefinedBy:
            if rel.is_a('IfcRelDefinesByProperties'):
                property_set = rel.RelatingPropertyDefinition
                if property_set.is_a('IfcPropertySet') and property_set.Name == "Pset_LUKS":
                    wall_pset_luks = property_set
                    break # Found the Pset_LUKS for this wall

    # If the wall doesn't have Pset_LUKS, create a new one for it
    if not wall_pset_luks:
         wall_pset_luks = ifc_file.create_entity(
            "IfcPropertySet",
            str(uuid.uuid4()), # GlobalId (UUID)
            None, # OwnerHistory (placeholder)
            "Pset_LUKS", # Name
            None, # Description
            [] # Start with an empty list of properties
        )
         # Create a relationship to define the property set for this wall
         ifc_file.create_entity(
            "IfcRelDefinesByProperties",
            str(uuid.uuid4()), # GlobalId (UUID)
            None, # OwnerHistory (placeholder)
            "Property definition for wall", # Name
            None, # Description
            [wall], # RelatedObjects
            wall_pset_luks # RelatingPropertyDefinition
        )

    # --- Logic to find the Building Storey and create GeschossID property ---
    geschossid_value = "n/a" # Default value if storey not found

    building_storey = find_building_storey(wall)

    if building_storey:
        if hasattr(building_storey, 'Name') and building_storey.Name:
            geschossid_value = building_storey.Name # Use the storey name as GeschossID
        elif hasattr(building_storey, 'LongName') and building_storey.LongName:
             geschossid_value = building_storey.LongName # Use LongName if Name is empty

    # --- End of GeschossID logic ---


    # Add the properties to this wall's Pset_LUKS if they don't exist
    properties_to_add_to_wall_pset = []

    # Check if ArealID property already exists in this wall's Pset_LUKS
    arealid_prop_exists = False
    for prop in wall_pset_luks.HasProperties:
        if prop.is_a('IfcPropertySingleValue') and prop.Name == 'ArealID':
            arealid_prop_exists = True
            break
    if not arealid_prop_exists:
        properties_to_add_to_wall_pset.append(property_arealid_template) # Use the template


    # Check if GebaeudeID property already exists in this wall's Pset_LUKS
    gebaeudeid_prop_exists = False
    for prop in wall_pset_luks.HasProperties:
        if prop.is_a('IfcPropertySingleValue') and prop.Name == 'GebaeudeID':
            gebaeudeid_prop_exists = True
            break
    if not gebaeudeid_prop_exists:
        properties_to_add_to_wall_pset.append(property_gebaeudeid_template) # Use the template


    # Check if Unterhaltsrelevant property already exists in this wall's Pset_LUKS
    unterhaltsrelevant_prop_exists = False
    for prop in wall_pset_luks.HasProperties:
        if prop.is_a('IfcPropertySingleValue') and prop.Name == 'Unterhaltsrelevant':
            unterhaltsrelevant_prop_exists = True
            break
    if not unterhaltsrelevant_prop_exists:
        properties_to_add_to_wall_pset.append(property_unterhaltsrelevant_template) # Use the template


    # Check if GeschossID property already exists in this wall's Pset_LUKS
    geschossid_prop_exists = False
    for prop in wall_pset_luks.HasProperties:
        if prop.is_a('IfcPropertySingleValue') and prop.Name == 'GeschossID':
            geschossid_prop_exists = True
            break
    # Add the GeschossID property instance if it doesn't exist
    if not geschossid_prop_exists:
        # Create the GeschossID property instance here to use the determined geschossid_value
        property_geschossid_instance = ifc_file.create_entity(
            "IfcPropertySingleValue",
            "GeschossID",
            None,  # Description is optional
            ifc_file.create_entity("IfcText", geschossid_value),  # Nominal value is the storey name
            None  # Unit is optional
        )
        properties_to_add_to_wall_pset.append(property_geschossid_instance) # Use the instance


    # Add the collected properties to the wall's Pset_LUKS if there are any to add
    if properties_to_add_to_wall_pset:
        # Create a new tuple with existing properties and the new ones
        wall_pset_luks.HasProperties = tuple(list(wall_pset_luks.HasProperties) + properties_to_add_to_wall_pset)
        modified_wall_count += 1


print(f"\nProcessed {len(walls_without_arealid)} walls. Added/updated Pset_LUKS on {modified_wall_count} walls.")

# Construct the output filename
base_name, ext = os.path.splitext(input_filename)
output_filename = f"{base_name}_modified{ext}"

# Save the modified IFC file
ifc_file.write(output_filename)
print(f"IFC created: {output_filename}")


Processed 985 walls. Added/updated Pset_LUKS on 985 walls.
IFC created: LU32_AR_MD_ME_LEITM_modified.ifc
